# CropDoc GB: Colab GPU runs (thin wrapper around `make` targets)

**Runtime → Change runtime type → T4 GPU.** This notebook:
1. mounts Google Drive (checkpoints and results survive disconnects),
2. gets the code (your GitHub repo or the zip on Drive),
3. downloads the data (no API keys),
4. runs the experiment matrix block by block (`make sweep`). **Finished runs are skipped and
   interrupted runs resume**, so after a disconnect just run the cells again,
5. aggregates results (`make eval`) and copies `results/` to Drive.

Estimated cost (planning estimates, replaced by measured epoch times in the logs): block A ≈ 3 GPU-h,
B ≈ 15.5, C ≈ 3.5, D ≈ 2.6, E ≈ 1.3. Free Colab gives a few GPU-hours per day, so run one block per session.

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime → Change runtime type → T4 GPU"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cropdoc-gb'   # checkpoints + results are kept here
import os; os.makedirs(DRIVE, exist_ok=True)

## Get the code
Option A: clone your GitHub repository (edit the URL; a private repo needs a token).
Option B: put `cropdoc-gb-complete.zip` in `MyDrive/cropdoc-gb/` and unzip it.

In [ ]:
REPO_URL = ''   # e.g. 'https://github.com/<you>/cropdoc-gb.git'; leave empty to use the zip on Drive
%cd /content
if REPO_URL:
    !git clone -q $REPO_URL cropdoc-gb
else:
    !unzip -q -o {DRIVE}/cropdoc-gb-complete.zip -d /content
%cd /content/cropdoc-gb

In [ ]:
# Colab already has a GPU build of torch/torchvision: install everything else with the project's pins.
!pip install -q -r ml/requirements-colab.txt
import torch, timm; print('torch', torch.__version__, '| timm', timm.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Data on the fast local disk (~2.5 GB): PlantVillage colour + leaf masks + PlantDoc. Splits are committed in ml/splits.
!python scripts/download_data.py --plantdoc
!PYTHONPATH=ml/src python -m cropdoc.data.build_splits --manifest-only

In [ ]:
# Runs (checkpoints, logits, metrics) go to Drive, so a disconnect loses at most one epoch.
os.environ['CROPDOC_RUNS'] = f'{DRIVE}/runs'
!make sweep-status

## Block A first: how to handle class imbalance
After it finishes, compare `imb_none`, `imb_weighted_loss` and `imb_balanced_sampler` on **validation** macro-F1
(`make eval`, then `results/sweep/summary.csv`). If one of the reweighting options wins, set
`data.class_balance` in `ml/configs/base.yaml` before running blocks B–E, and record it in `docs/decisions.md`.

In [ ]:
!make sweep SWEEP_ARGS="--block A"

In [ ]:
!make sweep SWEEP_ARGS="--block C"   # leakage: naive split (3 models x 3 seeds)

In [ ]:
!make sweep SWEEP_ARGS="--block B"   # model x augmentation (36 runs); safe to re-run after disconnects

In [ ]:
!make sweep SWEEP_ARGS="--block E"   # unseen-disease model
!make sweep SWEEP_ARGS="--block D"   # stretch: strong + background randomization

## Aggregate, save, and (maybe) promote a better model
`make eval` writes `results/sweep/` (tidy table, plot, pre-registered comparisons with bootstrap CIs).
`make promote RUN=<run_id>` swaps a run into `models/active/` **only if** it beats the current model's
group-aware test macro-F1. Afterwards download `results/` and `models/active/` from Drive into your repo, then run
`make report` and `make package` on your laptop.

In [ ]:
!make eval
!mkdir -p {DRIVE}/results && cp -r results/* {DRIVE}/results/
!cat results/sweep/summary.md

In [ ]:
# Optional: promote the best group-aware run (pick its run id from results/sweep/runs.csv)
# !make promote RUN=s0_xxxxxxxxxx && cp -r models/active {DRIVE}/models_active